# Avance 2. Lectura, perfilamiento y preparacion de datos
**Proyecto:** SuperMarketMundial · **Equipo:** Equipo BI SuperMarket
**Integrantes:** Francisco Aramis Alcantara Vivas · Brayan Perez Nava · Joseline Ariana Marcial Espinoza

## 3.1 Continuidad con el avance 1

**Pregunta de negocio.** ¿A partir de que nivel de descuento la venta deja de ser rentable, y ese punto es el mismo en todas las categorias, subcategorias y mercados?

**Quien usa la informacion.** La Gerencia de Marketing, que decide los descuentos, y la Gerencia de Logistica, que revisa los costos de envio.

**Decision.** Fijar un descuento maximo por subcategoria y mercado para dejar de vender con perdida, y revisar si conviene descontar menos en los meses en que se vende mas.

**Alcance.** Ventas de Estados Unidos, Latinoamerica, Europa y Asia-Pacifico, de enero de 2011 a diciembre de 2014, revisadas mes por mes.

**Fuente.** Archivo *Ventas Supermercado* publicado por Rodrigo Ayala en GitHub (https://github.com/RodrigoAyalaDA/Analisis-Ventas-Excel).

**Columnas indispensables.** Descuento, Ventas, Utilidad, Categoria, Subcategoria, Mercado, Fecha Salida y Costo Envío.

**Subpreguntas**
1. ¿Como cambia el margen conforme aumenta el descuento?
2. ¿El limite es el mismo para todas las categorias y subcategorias?
3. ¿El limite es el mismo en todos los mercados?
4. ¿Que porcentaje de las ventas ocurre por encima del limite y cuanta utilidad se pierde ahi?
5. ¿Como cambian las ventas, los descuentos y las perdidas mes a mes?

**Ajustes por la retroalimentacion del avance 1**

| Observacion | Quien la hizo | Que cambiamos | Evidencia |
|---|---|---|---|
| Faltaban las subpreguntas | Profesora | Se agregaron cinco subpreguntas | Lista de arriba y avance 1 corregido, seccion 3.3 |
| Acotar a zonas con una cantidad de ventas parecida y agregar el analisis por mes | Profesora | Se trabaja con cuatro mercados y se crea una columna con el mes | Secciones 3.4 y 3.5 de este notebook |
| Analizar tambien la subcategoria | Profesora | La subcategoria es el nivel donde se propondra el descuento maximo | Avance 1 corregido, secciones 3.3 y 3.7 |
| La version entregada decia que el periodo era 2011–2015 | Equipo | Las ventas van de 2011 a 2014. El 2015 solo aparece en fechas de entrega | Seccion 3.2 de este notebook |
| Usabamos regiones | Equipo | Ahora se compara por mercado | Seccion 3.3 de este notebook |

## 3.2 Lectura reproducible y verificacion inicial

Leemos el Excel con Python y comprobamos que se leyeron el archivo y las hojas correctas antes de revisar los datos. Tambien registramos de donde viene el archivo, que version es, que hojas se leyeron y con que parametros, para que cualquiera pueda repetir la lectura y obtener el mismo resultado.

### Archivo y huella

In [ ]:
# Nos aseguramos con la huella que el archivo sea siempre el mismo y que
# no lo hayamos modificado por accidente
import hashlib #libreria para la huella
from pathlib import Path
import pandas as pd
from google.colab import files

subido = files.upload()                 # abre la ventana para elegir el Excel
ruta = Path(next(iter(subido)))         # guardar la ruta

huella = hashlib.sha256(ruta.read_bytes()).hexdigest() #generamos una huella unica del archivo
# y la guardamos en huella

#Imprime la informacion del archivo
print("Archivo:", ruta.name)
print("Tamaño:", round(ruta.stat().st_size / 1_000_000, 1), "MB")
print("Huella SHA-256 (inicio):", huella[:16])

libro = pd.ExcelFile(ruta) # cargamos la ruta del archivo en libro
print("Hojas del libro:", libro.sheet_names) #imprimir nombres de las hojas del excel


Saving Ventas Supermercado- Rodrigo Ayala.xlsx to Ventas Supermercado- Rodrigo Ayala (1).xlsx
Archivo: Ventas Supermercado- Rodrigo Ayala (1).xlsx
Tamaño: 20.2 MB
Huella SHA-256 (inicio): a599dc2cf5d00385
Hojas del libro: ['Guía del Archivo', 'Dashboard', 'Analisis', 'Ventas Supermercado', 'Regiones', 'Productos']


### Lectura de las hojas y verificacion

In [ ]:
# Leemos solo las hojas con datos que vamos a ocupar para nuestro analisis
hojas = pd.read_excel( #las guardamos en hojas
    libro,
    sheet_name=["Ventas Supermercado", "Regiones", "Productos"],
    header=0,   # Obtenemoos la primera fila de cada hoja porque ahi estan las columnas
)
#separamos las hojas en cada variable
ventas = hojas["Ventas Supermercado"]
regiones = hojas["Regiones"]
productos = hojas["Productos"]

#Comprobamos que se leyeron correctamente
# Revisamoos que el archivo tenga las mismas filas y columnas que registramos en el avance 1
print("Ventas Supermercado:", type(ventas).__name__, ventas.shape, "coincide el tamanho con ventas?", ventas.shape == (51290, 35))
print("Regiones:", type(regiones).__name__, regiones.shape, "coincide el tamanho con regiones?", regiones.shape == (152, 3))
print("Productos:", type(productos).__name__, productos.shape, "coincide el tamanho con productos?", productos.shape == (10768, 4))

print("\nColumnas de ventas:", ventas.columns.tolist()) #imprimimos las columnas de ventas
print("Columnas de regiones:", regiones.columns.tolist()) #imprimimos las columnas de regiones
print("Columnas de productos:", productos.columns.tolist()) #imprimimos las columnas de productos
print("\nPrimera venta:", ventas["Fecha Salida"].min().date())
print("Última venta:", ventas["Fecha Salida"].max().date())
ventas.head(3)#imprimimos las pprimeras 3 filas de ventas

Ventas Supermercado: DataFrame (51290, 35) coincide el tamanho con ventas? True
Regiones: DataFrame (152, 3) coincide el tamanho con regiones? True
Productos: DataFrame (10768, 4) coincide el tamanho con productos? True

Columnas de ventas: ['Número Venta', 'ID Venta', 'Mes Salida', 'Dia Salida', 'Año Salida', 'Fecha Salida', 'Dia Semana Fecha Salida', 'Mes Entrega', 'Dia Entrega', 'Año Entrega', 'Fecha Entrega', 'Dia Semana Fecha Entrega', 'Dias Transcurridos', 'Dias laborables transcurridos', 'Método Envio', 'ID Cliente', 'Número Cliente', 'Nombre Cliente', 'Segmento', 'Ciudad', 'Estado', 'País', 'Mercado', 'Region', 'ID Producto', 'Categoria', 'Subcategoria', 'Nombre del producto', 'Ventas', 'Cantidad', 'Descuento', 'Utilidad', 'Ganancia', 'Costo Envío', 'Prioridad Envio']
Columnas de regiones: ['País', 'Mercado', 'Región']
Columnas de productos: ['ID Producto', 'Categoría', 'Sub-Categoría', 'Nombre Producto']

Primera venta: 2011-01-01
Última venta: 2014-12-31


,Número Venta,ID Venta,Mes Salida,Dia Salida,Año Salida,Fecha Salida,Dia Semana Fecha Salida,Mes Entrega,Dia Entrega,Año Entrega,Fecha Entrega,Dia Semana Fecha Entrega,Dias Transcurridos,Dias laborables transcurridos,Método Envio,ID Cliente,Número Cliente,Nombre Cliente,Segmento,Ciudad,Estado,País,Mercado,Region,ID Producto,Categoria,Subcategoria,Nombre del producto,Ventas,Cantidad,Descuento,Utilidad,Ganancia,Costo Envío,Prioridad Envio
0,2040,AL-2011-2040,1,1,2011,2011-01-01,6,1,6,2011,2011-01-06,4,5,4,Standard Class,TB-11280,11280,Toby Braunhardt,Consumer,Constantine,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408.300,2,0.0,106.140,Ganancia,35.46,Medium
1,47883,AU-2011-47883,1,1,2011,2011-01-01,6,1,8,2011,2011-01-08,6,7,5,Standard Class,JH-15985,15985,Joseph Holt,Consumer,Wagga Wagga,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.366,3,0.1,36.036,Ganancia,9.72,Medium
2,1220,HU-2011-1220,1,1,2011,2011-01-01,6,1,5,2011,2011-01-05,3,4,3,Second Class,AT-735,735,Annie Thurman,Consumer,Budapest,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66.120,4,0.0,29.640,Ganancia,8.17,High


### Contrato de lectura

| Elemento | Registro |
|---|---|
| Fuente | *Ventas Supermercado*, publicado por Rodrigo Ayala en GitHub (https://github.com/RodrigoAyalaDA/Analisis-Ventas-Excel).|
| Version y fecha de obtencion | La fuente no declara version. Se consulto el 15 de septiembre de 2026 (avance 1, seccion 3.5). La copia usada pesa 20.2 MB y su huella SHA-256 empieza con `a599dc2cf5d00385` |
| Ubicacion | En Colab el archivo se sube con `files.upload()`. En el repositorio esta en `https://github.com/Kaedee2/Proyecto-superMarketGlobal-BI/blob/main/Ventas%20Supermercado-%20Rodrigo%20Ayala.xlsx` sin modificaciones |
| Formato y recurso | Libro de Excel (.xlsx) con seis hojas. Se leen Ventas Supermercado (la principal), Regiones y Productos. No se leen Guía del Archivo, que solo describe las hojas, tampoco Dashboard, ni Analisis, que tiene resumenes hechos por el autor |
| Funcion y parametros | `pd.ExcelFile(ruta)` para ver las hojas y `pd.read_excel(libro, sheet_name=[...], header=0)` para leerlas por su nombre, con la primera fila como encabezado |
| Objeto esperado | Un DataFrame por cada hoja, ventas con 51,290 filas y 35 columnas, Regiones con 152 y 3, y Productos con 10,768 y 4 |
| Comprobaciones | Las tres hojas tienen las filas y columnas registradas en el avance 1. La primera venta es del 1 de enero de 2011 y la ultima del 31 de diciembre de 2014 |
| Advertencias | Las hojas nombran distinto las mismas columnas: `Region` en ventas y `Región` en Regiones, `Categoria` y `Subcategoria` en ventas, y `Categoría` y `Sub-Categoría` en Productos. La fuente no tiene diccionario de datos ni licencia |

**Resultado de la lectura.** Se leyeron las hojas correctas. Las tres tienen las mismas filas y columnas que registramos en el avance 1 y las ventas van del 1 de enero de 2011 al 31 de diciembre de 2014. Que la lectura saliera bien no quiere decir que los datos esten limpios. Eso lo revisamos en la seccion 3.3.

## 3.3 Inspeccion y perfilamiento

Antes de corregir cualquier cosa revisamos como vienen los datos. En esta seccion vemos la estructura de la tabla, el tipo de cada columna, si hay datos vacios, cuantos valores distintos tiene cada columna, que registros se repiten y que valores son muy altos o muy bajos.

Esto sirve para saber que problemas tiene el archivo y cuales pueden cambiar la respuesta a la pregunta de negocio. Aqui solo se detectan los problemas. Que se corrige y por que lo decidimos en la seccion 3.4.

### Que representa una fila y clave candidata

> **Guia:** Identifiquen que representa una fila y que columna o combinacion podria reconocer la granularidad esperada. El indice automatico de Pandas no constituye por si mismo una clave de negocio.


In [ ]:
# Un pedido (ID Venta) puede tener varios productos, parecido a ir a comprar al super y que te den un ticket que tiene un id al haber comprado varias cosas .
# Por ejemplo, el pedido BA-2011-30733 tiene 4 filas, una por producto.
display(ventas[ventas["ID Venta"] == "BA-2011-30733"][["ID Venta", "ID Producto", "Nombre del producto", "Cantidad", "Ventas"]])

# Que columnas identifican cada fila?
print("Total de Filas:", len(ventas)) #Cuenta cuantas filas tiene ventas
print("Pedidos distintos (solo ID Venta):", ventas["ID Venta"].nunique()) #cuenta cuantas tickets de venta hay
#por ejemplo BA-2011-30733 lo cuenta como una venta aunque hayan 4 productos con ese id de venta.
print("Combinaciones distintas de pedido y producto:", len(ventas[["ID Venta", "ID Producto"]].drop_duplicates()))
# quita las filas en donde id venta e id producto se repiten al mismo tiempo
#el resultado es que nos da filas en donde hay repetidos

,ID Venta,ID Producto,Nombre del producto,Cantidad,Ventas
6,BA-2011-30733,TEC-CO-10002316,"Brother Personal Copier, Laser",2,285.78
14,BA-2011-30733,OFF-SU-10000484,"Kleencut Ruler, High Speed",3,40.68
17,BA-2011-30733,OFF-BI-10003012,"Wilson Jones Hole Reinforcements, Economy",5,22.65
18,BA-2011-30733,OFF-LA-10001292,"Smead File Folder Labels, Adjustable",3,20.34


Total de Filas: 51290
Pedidos distintos (solo ID Venta): 25086
Combinaciones distintas de pedido y producto: 51252


**Conclusion de que representa una fila y la clave:**

* **¿Que representa una fila?** Un producto dentro de un pedido, como cada linea de un ticket del super.
* **¿Que columnas identifican cada fila?** La combinacion de `ID Venta` + `ID Producto`.
  * `ID Venta` solo no sirve porque se repite por cada producto que tenga el pedido.
  * `ID Producto` solo tampoco sirve porque el mismo producto se vende muchas veces a distintos clientes.
* **¿Podemos usar ID Venta + ID Producto como clave?** Casi. Hay 51,290 filas y 51,252 combinaciones distintas, y la diferencia son 38 parejas (76 filas) donde se repite el mismo pedido y producto. Por eso por ahora la dejamos como clave candidata, y pasa a ser la clave definitiva cuando decidimos que hacer con esas parejas (seccion 3.4).

### Estructura

> **Guia:** Filas, columnas, nombres, registros iniciales y finales, y resumen de info().

In [ ]:
# Registros iniciales y finales:
# Para comprobar que el archivo se leyp completo y no se corto al final
print("Primeras 3 filas:")
display(ventas.head(3))
print("Últimas 3 filas:")
display(ventas.tail(3))

# Resumen de la tabla: filas, columnas, nombres, tipo de cada columna y cuantos valores tiene
ventas.info()

Primeras 3 filas:


,Número Venta,ID Venta,Mes Salida,Dia Salida,Año Salida,Fecha Salida,Dia Semana Fecha Salida,Mes Entrega,Dia Entrega,Año Entrega,...,Categoria,Subcategoria,Nombre del producto,Ventas,Cantidad,Descuento,Utilidad,Ganancia,Costo Envío,Prioridad Envio
0,2040,AL-2011-2040,1,1,2011,2011-01-01,6,1,6,2011,...,Office Supplies,Storage,"Tenex Lockers, Blue",408.300,2,0.0,106.140,Ganancia,35.46,Medium
1,47883,AU-2011-47883,1,1,2011,2011-01-01,6,1,8,2011,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.366,3,0.1,36.036,Ganancia,9.72,Medium
2,1220,HU-2011-1220,1,1,2011,2011-01-01,6,1,5,2011,...,Office Supplies,Storage,"Tenex Box, Single Width",66.120,4,0.0,29.640,Ganancia,8.17,High


Últimas 3 filas:


,Número Venta,ID Venta,Mes Salida,Dia Salida,Año Salida,Fecha Salida,Dia Semana Fecha Salida,Mes Entrega,Dia Entrega,Año Entrega,...,Categoria,Subcategoria,Nombre del producto,Ventas,Cantidad,Descuento,Utilidad,Ganancia,Costo Envío,Prioridad Envio
51287,110527,NI-2015-110527,12,31,2014,2014-12-31,3,1,2,2015,...,Office Supplies,Labels,"Hon Color Coded Labels, 5000 Label Set",26.400,3,0.0,12.3600,Ganancia,0.35,Medium
51288,114783,ME-2015-114783,12,31,2014,2014-12-31,3,1,6,2015,...,Office Supplies,Labels,"Hon Legal Exhibit Labels, Alphabetical",7.120,1,0.0,0.5600,Ganancia,0.20,Medium
51289,156720,UN-2015-156720,12,31,2014,2014-12-31,3,1,4,2015,...,Office Supplies,Fasteners,Bagged Rubber Bands,3.024,3,0.2,-0.6048,Perdida,0.17,Medium


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 35 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Número Venta                   51290 non-null  int64         
 1   ID Venta                       51290 non-null  object        
 2   Mes Salida                     51290 non-null  int64         
 3   Dia Salida                     51290 non-null  int64         
 4   Año Salida                     51290 non-null  int64         
 5   Fecha Salida                   51290 non-null  datetime64[ns]
 6   Dia Semana Fecha Salida        51290 non-null  int64         
 7   Mes Entrega                    51290 non-null  int64         
 8   Dia Entrega                    51290 non-null  int64         
 9   Año Entrega                    51290 non-null  int64         
 10  Fecha Entrega                  51290 non-null  datetime64[ns]
 11  Dia Semana Fech

### Tipos

> **Guia:** Tipo detectado por columna y dudas sobre su correspondencia con el significado.

In [ ]:
# Tipos en cada una de las 35 columnas
print(ventas.dtypes.value_counts(), "\n")  # cuantas columnas hay de cada tipo
ventas.dtypes

object            16
int64             13
float64            4
datetime64[ns]     2
Name: count, dtype: int64 



,0
Número Venta,int64
ID Venta,object
Mes Salida,int64
Dia Salida,int64
Año Salida,int64
Fecha Salida,datetime64[ns]
Dia Semana Fecha Salida,int64
Mes Entrega,int64
Dia Entrega,int64
Año Entrega,int64


**Dudas sobre los tipos:**

* **Columnas de codigo guardadas como numero.** `Número Venta`, `Número Cliente`, `Mes Salida`, `Dia Salida`, `Año Salida`, `Dia Semana Fecha Salida` y las mismas cuatro de entrega vienen como `int64`. Son numeros, pero no tiene sentido sumarlos ni sacarles promedio, porque sirven para identificar algo o para ubicar una fecha (por ejemplo, en `Dia Semana` el 1 es lunes y el 7 es domingo).
* **Ganancia es texto.** `Ganancia` viene como `object` porque solo dice "Ganancia" o "Perdida", y eso ya lo dice el signo de `Utilidad`.
* **Lo demas esta bien.** Las fechas ya vienen como fecha (`datetime64`), `Cantidad` y los dias transcurridos son conteos enteros, y Ventas, Descuento, Utilidad y Costo Envío son numeros con decimales.

### Faltantes

> **Guia:** Valores presentes, cantidad y porcentaje de ausencias por columna.

In [ ]:
# tabla con valores presentes, nulos y porcentaje de nulos en cada una de las 35 columnas
faltantes = pd.DataFrame({
    "valores_presentes": ventas.notna().sum(),
    "valores_nulos": ventas.isna().sum(),
})
#Calculamos el porcentaje de valores nulos (con un decimal)
faltantes["porcentaje_nulos"] = (faltantes["valores_nulos"] / len(ventas) * 100).round(1)
# Cuantas columnas tienen al menos un dato vacio
print("Columnas con algún nulo:", (faltantes["valores_nulos"] > 0).sum(), "de", len(faltantes))
#imprimimos la tabla
faltantes

Columnas con algún nulo: 0 de 35


,valores_presentes,valores_nulos,porcentaje_nulos
Número Venta,51290,0,0.0
ID Venta,51290,0,0.0
Mes Salida,51290,0,0.0
Dia Salida,51290,0,0.0
Año Salida,51290,0,0.0
Fecha Salida,51290,0,0.0
Dia Semana Fecha Salida,51290,0,0.0
Mes Entrega,51290,0,0.0
Dia Entrega,51290,0,0.0
Año Entrega,51290,0,0.0


In [ ]:
# Revisamos que ninguna celda de texto este vacia o diga "N/A", "null", "-", etc.
marcas = ["", "N/A", "NA", "null", "NULL", "-", "None", "nan"]
total = 0
for col in ventas.select_dtypes(include="object").columns:
    total += ventas[col].astype(str).str.strip().isin(marcas).sum()
print("Celdas de texto vacias o con marcas de ausencia:", total)

Celdas de texto vacias o con marcas de ausencia: 0


### Resultado de datos faltantes

* **Resultado:** No hay datos faltantes en ninguna columna. Las 35 columnas tienen sus 51,290 registros completos.
* **¿Con que lo comprobamos?**
  1. Con `isna().sum()`, que dio 0 vacios en todas las columnas.
  2. Revisando que ninguna celda de texto estuviera en blanco o dijera "N/A", "null" o "-", porque eso tambien seria un dato faltante escondido. Tambien dio 0.
* **¿Que quiere decir para el analisis?** Como el archivo viene completo, no hay que rellenar ni quitar nada por datos vacios.

### Diversidad y frecuencias

> **Guia:** Valores distintos y frecuencias relevantes, conservando explicitamente las ausencias cuando corresponda.

In [ ]:
#Diversidad:  Cuantos valores distintos tiene cada una de las 35 columnas
ventas.nunique(dropna=True)

,0
Número Venta,18541
ID Venta,25086
Mes Salida,12
Dia Salida,31
Año Salida,4
Fecha Salida,1430
Dia Semana Fecha Salida,7
Mes Entrega,12
Dia Entrega,31
Año Entrega,5


In [ ]:
# Frecuencias de las columnas que usa la pregunta (quiere decir cuantas veces se repite cada dato en las columnas impoetantes).
# dropna=False hace que tambien se cuenten los vacios si los hubiera.
print("Cuantas ventas hubo en cada mercado")
print(ventas["Mercado"].value_counts(dropna=False), "\n")

print("Cuantas ventas hubo en cada categoria")
print(ventas["Categoria"].value_counts(dropna=False), "\n")

print("Cuantas ventas hubo por subcategoria")
print(ventas["Subcategoria"].value_counts(dropna=False), "\n")

print("Cuantas veces se aplico cada descuento (del menor al  con sort_index)")
print(ventas["Descuento"].value_counts(dropna=False).sort_index())

Cuantas ventas hubo en cada mercado
Mercado
APAC      11001
LATAM     10294
US         9994
EU         9730
EMEA       5300
Africa     4587
Canada      384
Name: count, dtype: int64 

Cuantas ventas hubo en cada categoria
Categoria
Office Supplies    31273
Technology         10141
Furniture           9876
Name: count, dtype: int64 

Cuantas ventas hubo por subcategoria
Subcategoria
Binders        6155
Storage        5059
Art            4883
Paper          3538
Chairs         3434
Phones         3357
Furnishings    3170
Accessories    3075
Labels         2603
Envelopes      2435
Supplies       2425
Fasteners      2420
Bookcases      2411
Copiers        2223
Appliances     1755
Machines       1486
Tables          861
Name: count, dtype: int64 

Cuantas veces se aplico cada descuento (del menor al  con sort_index)
Descuento
0.000    29009
0.002      461
0.070      150
0.100     4068
0.150      541
0.170      735
0.200     4998
0.202       41
0.250      198
0.270      388
0.300      340
0.

**Variedad de datos (`nunique`):**
  * Notamos que Columnas como `Categoria` (3 valores) y `Mercado` (7 valores) tienen muy pocas opciones.
  * Por el contrario, `ID Producto` cuenta con 10,292 valores distintos, lo que muestra un catalogo de articulos muy grande.

**Conteo de repeticiones en columnas clave:**
  * **Mercado:** Las ventas se concentran principalmente en 4 lugares: APAC (11,001), LATAM (10,294), US (9,994) y EU (9,730), con cerca de 10,000 ventas cada uno. Los otros tres (EMEA, Africa y Canada) tienen mucho menos ventas. Esto confirma que comparar solo los 4 mercados principales hace la comparacion mas equilibrada porque tienen cantidades parecidas de ventas.
  * **Categoria:** La tienda vende 3 tipos de articulos: papeleria y oficina (*Office Supplies*: 31,273 ventas), tecnologia (*Technology*: 10,141 ventas) y muebles (*Furniture*: 9,876 ventas). Mas de la mitad de las ventas son articulos de oficina.
  * **Subcategoria:** De las 17 subcategorias disponibles, lo que mas compra la gente son carpetas (*Binders*: 6,155) y organizadores o cajas (*Storage*: 5,059). Lo que menos se vende son mesas (*Tables*: 861) y maquinas (*Machines*: 1,486).
  * **Descuento:** En 29,009 ventas (mas de la mitad de todas las ventas) no se otorgo ningun descuento (`0.0`). Cuando si se aplica un descuento, casi siempre se usan cifras cerradas como 10%, 20% o 40%.
  Notamos que hay descuentos raros como 0.002

#### Descuentos poco comunes

> **Guia:** Seleccionen y filtren registros con condiciones explicitas, conservando identificadores y contexto para interpretar cada caso. Documenten que poblacion representa un conteo despues de filtrar.

Notamos que hay descuentos raros como 0.002, 0.402 que podrian darnos problemas mas adelante

In [ ]:
# Separamos las ventas con descuentos poco comunes para ver cuantas son y de donde vienen
raros = [0.002, 0.202, 0.402, 0.602]
# round(3) deja el descuento con 3 decimales para comparar bien porque luego quedan decimales de mas como 2.002000000
descuentos_raros = ventas[ventas["Descuento"].round(3).isin(raros)]

print("Ventas con descuento poco común:", len(descuentos_raros), "de", len(ventas))
print(descuentos_raros["Mercado"].value_counts()) #imprimimos cuantas y de que mercado son
descuentos_raros[["ID Venta", "País", "Mercado", "Subcategoria", "Ventas", "Descuento", "Utilidad"]].head()

Ventas con descuento poco común: 629 de 51290
Mercado
LATAM    629
Name: count, dtype: int64


,ID Venta,País,Mercado,Subcategoria,Ventas,Descuento,Utilidad
220,CU-2011-133662,Cuba,LATAM,Copiers,231.97512,0.002,43.69512
245,HO-2011-135104,Honduras,LATAM,Copiers,253.98256,0.402,-128.29744
398,BR-2011-159436,Brazil,LATAM,Copiers,153.52452,0.602,-162.79548
401,HO-2011-150987,Honduras,LATAM,Copiers,230.67252,0.402,-85.64748
402,BR-2011-159436,Brazil,LATAM,Copiers,114.76728,0.602,-61.15272


Con head obtenemos las primeras filas, notamos que en donde se repite el error de agregar 0.002 es en las filas que pertenecen a LATAM y ademas estan en la subcategoria Copiers
como head solo nos da las primeras filas, contamos cuantas veces se repite Copiers para ver si en efecto todas las filas de esa subcategoria tienen este problema

In [ ]:
# Contamos las subcategorias y vemos que es la misma cantidad que los mercados
print(descuentos_raros["Subcategoria"].value_counts())

# Cuantas copiadoras se vendieron en LATAM en total? confirmamos que todos los errores de descuento corresponden a las copiadoras
#que se vendieron en LATAM
copiers_latam = ventas[(ventas["Mercado"] == "LATAM") & (ventas["Subcategoria"] == "Copiers")]
print("Copiers vendidas en LATAM:", len(copiers_latam))

Subcategoria
Copiers    629
Name: count, dtype: int64
Copiers vendidas en LATAM: 629


**Resultado.** Las 629 ventas con descuento raro (0.002, 0.202, 0.402 y 0.602) son todas las copiadoras que se vendieron en LATAM, y ningun otro producto ni mercado tiene esos valores. Como pasa en todo un grupo y no en ventas sueltas, no parece un error de dedo al azar, pero el archivo no dice si es una regla de descuento o un error. Despues lo revisamos con la profesora y nos indico tomarlos como error de captura y redondearlos (regla 4 de la seccion 3.4).

### Repeticiones

> **Guia:** Conteo de duplicados exactos y revision de repeticiones segun la clave candidata.

Revisamos registros repetidos. Puede darse el caso de que dos registros aparentemente pueden parecer dos copias exactas pero en realidad sean por ejemplo 2 productos iguales donde aparentemente todo sea igual pero cambie por ejemplo la fecha en que se compro el producto. Entonces aunque aparentemente parezca un duplicado, en realidad son compras distintas.

**Por ello vamos a identificar duplicados para saber cuales si son duplicados exactos y cuales no**

In [ ]:
# Identificamos los duplicados exactos en las 35 columnas de ventas
print("Filas exactamente repetidas:", ventas.duplicated().sum()) #Sumar las filas duplicadas

#Para asegurarnos de que las filas no sean repetidas usamos la combinacion de id Venta e ID producto,
# Esta combinacion no puede repetirse porque ambas cambian, de esta manera podemos verificar los duplicados
clave = ["ID Venta", "ID Producto"]
repetidas = ventas[ventas.duplicated(subset=clave, keep=False)]
print("Filas que repiten pedido y producto:", len(repetidas), "de", len(ventas))
repetidas.sort_values(clave)[clave + ["Cantidad", "Ventas", "Descuento", "Utilidad", "Costo Envío"]].head(6)
#Queremos identificar esta clase de filas en donde estos ids son los mismos pero cambian algunas columnas porque no son
#duplicados exactos, esto quiere decir que puede darse el caso de que las filas se parezcan a proposito por ejemplo,
#Si solo cambia el costo de envio puede que sea que por ejemplo el pedido no pudo enviarse en un solo envio, entonces
#agregaron otro envio y por ello tiene otro precio de envio. Es por ello que tenemos que decidir que hacer con estas filas

#Revisamos los registros repetidos de dos formas. Primero buscamos duplicados exactos, es decir, filas identicas en las 35 columnas.
 #Despues buscamos filas con el mismo pedido y el mismo producto (ID Venta + ID Producto), porque cada fila deberia ser un producto dentro de un pedido y esa combinacion no deberia repetirse. Si se repite, puede ser una captura doble o dos compras del mismo producto en el mismo pedido.

Filas exactamente repetidas: 0
Filas que repiten pedido y producto: 76 de 51290


,ID Venta,ID Producto,Cantidad,Ventas,Descuento,Utilidad,Costo Envío
32830,AL-2014-4840,TEC-MEM-10002524,2,56.760,0.0,10.740,5.43
32840,AL-2014-4840,TEC-MEM-10002524,2,56.760,0.0,10.740,2.77
773,AU-2011-54176,OFF-BI-10000168,3,15.390,0.1,2.520,0.70
774,AU-2011-54176,OFF-BI-10000168,3,15.390,0.1,2.520,0.64
11951,AU-2013-52804,OFF-AP-10000729,5,2367.360,0.1,604.860,196.63
11954,AU-2013-52804,OFF-AP-10000729,1,473.472,0.1,120.972,65.57


In [ ]:
# Que columnas cambian dentro de cada pareja?
# Se agrupa por pedido y producto, y se cuentan los valores distintos de cada columna.
# Si una columna tiene mas de 1 valor dentro de la pareja, quiere decir que cambia.
columnas = ["Cantidad", "Ventas", "Descuento", "Utilidad", "Costo Envío", "Fecha Salida", "ID Cliente"]
cambios = repetidas.groupby(clave)[columnas].nunique() > 1 #Agrupa las filas que tienen id venta e id producto iguales
# de esta manera forma 38 parejas, despues cuenta cuantos valores diferentes hay en las columnas de las parejas,
# si solo hay un valor distinto no cambian los valores de las columnas, esto quiere decir que son iguales
#si hay 2 o mas valores distintos quiere decir que son diferentes, guardamos

print("Parejas:", len(cambios)) #Muestra cuantas parejas tienen algun valor distinto en alguna de sus columnas
print("En cuántas parejas cambia cada columna:")
print(cambios.sum()) #Muestra en cuantas de las parejas cada columna es diferente, notamos que en todas las parejas
#El costo de envio es distinto, esto quiere decir que todas las parejas cambian en tener un costo de envio diferente
#Tambien quiere decir que esas 76 filas no son repetidas

Parejas: 38
En cuántas parejas cambia cada columna:
Cantidad        28
Ventas          28
Descuento        0
Utilidad        28
Costo Envío     38
Fecha Salida     3
ID Cliente       3
dtype: int64


### Resultado de las repeticiones

Al revisar si habia registros repetidos encontramos esto:

* **Filas exactamente iguales:** ninguna.
* **Mismo pedido y producto:** hay 38 parejas (76 filas) donde se repiten el `ID Venta` y el `ID Producto`. No son copias exactas, porque en todas cambia el `Costo Envío`, y en 28 tambien cambian la `Cantidad`, las `Ventas` y la `Utilidad`. Con el archivo no podemos saber si son dos capturas de la misma venta o un pedido que se mando en dos partes.
* **¿Que hacemos con ellas?** No las borramos. Por indicacion de la profesora, las 35 parejas que solo cambian en cantidad, ventas, utilidad y costo de envio las tomamos como un pedido mandado en dos partes y las juntamos en una fila (regla 6). Las 3 parejas donde cambian el cliente, la ciudad y la fecha las mandamos a pendientes (regla 5).

In [ ]:
#Ahora queremos ver si hay duplicados escondidos y que tengan cambios que hayamos pasado por alto como
#por ejemplo, que tenga una letra minuscula o espacios. los vamos a revisar en  id venta e id producto
#ejemplo "CA-2014-1" y "ca-2014-1 "

claves_limpias = pd.DataFrame({ #Hacemmos una copia de id venta e id producto de las 51,290 filas
#les quitamos los espacios y las pasamos a mayusculas
    "ID Venta": ventas["ID Venta"].str.strip().str.upper(),
    "ID Producto": ventas["ID Producto"].str.strip().str.upper(),
})

# contamos de las 51,290 filas los duplicados de id venta, id producto sin cambios (asi como vienen, sin mayus y sin quitar espacios)
# y los imprimimos
print("Repetidas tal como vienen:", ventas.duplicated(subset=clave, keep=False).sum())
# contamos ahora los id venta, id producto sin espacios y en mayusculsa
print("Repetidas con claves limpias:", claves_limpias.duplicated(keep=False).sum())
# ambos tienen 76 y 76. si por ejemplo en claves limpias hubieran salido 80, esto quiere decir que al cambiar a mayusculas
# y quitar espacios encontramos que habian mas filsa repetidas pero no fue el caso, entonces concluimos con que
#no hay duplicados ocultos y en efecto estos son los unicos duplicados que hay



Repetidas tal como vienen: 76
Repetidas con claves limpias: 76



* **Revision de textos (claves limpias):**
  * Al quitar espacios en blanco en las orillas y poner las letras en mayusculas, el resultado siguio siendo de 76 filas. Esto confirma que no hay otros duplicados escondidos por errores de dedo.

### Variables numericas

> **Guia:** Resumenes descriptivos y registros extremos que requieran interpretacion, cuando existan variables pertinentes.

In [ ]:
# Resumen de las columnas numericas que usa la pregunta
display(ventas[["Ventas", "Cantidad", "Descuento", "Utilidad", "Costo Envío"]].describe().round(2))

# Las 3 ventas con mas perdida y las 3 con mas ganancia, con su contexto
contexto = ["ID Venta", "Mercado", "Subcategoria", "Nombre del producto", "Ventas", "Descuento", "Utilidad"]
print("Mayores pérdidas:")
display(ventas.nsmallest(3, "Utilidad")[contexto])
print("Mayores ganancias:")
display(ventas.nlargest(3, "Utilidad")[contexto])

,Ventas,Cantidad,Descuento,Utilidad,Costo Envío
count,51290.00,51290.00,51290.00,51290.00,51290.00
mean,246.49,3.48,0.14,28.61,26.38
std,487.57,2.28,0.21,174.34,57.30
min,0.44,1.00,0.00,-6599.98,0.00
25%,30.76,2.00,0.00,0.00,2.61
50%,85.05,3.00,0.00,9.24,7.79
75%,251.05,5.00,0.20,36.81,24.45
max,22638.48,14.00,0.85,8399.98,933.57


Mayores pérdidas:


,ID Venta,Mercado,Subcategoria,Nombre del producto,Ventas,Descuento,Utilidad
43453,UN-2013-108196,US,Machines,Cubify CubeX 3D Printer Double Head Print,4499.985,0.7,-6599.9780
43136,TU-2013-9400,EMEA,Phones,"Motorola Smart Phone, Cordless",3085.344,0.6,-4088.3760
15886,UN-2014-168116,US,Machines,Cubify CubeX 3D Printer Triple Head Print,7999.980,0.5,-3839.9904


Mayores ganancias:


,ID Venta,Mercado,Subcategoria,Nombre del producto,Ventas,Descuento,Utilidad
8898,UN-2013-118689,US,Copiers,Canon imageCLASS 2200 Advanced Copier,17499.95,0.0,8399.9760
38848,UN-2014-140151,US,Copiers,Canon imageCLASS 2200 Advanced Copier,13999.96,0.0,6719.9808
29946,UN-2014-166709,US,Copiers,Canon imageCLASS 2200 Advanced Copier,10499.97,0.0,5039.9856


**Resultado de los valores extremos:**

* **Ventas.** Van de 0.44 a 22,638.48. La mitad de las ventas es de menos de 85.05, pero el promedio es 246.49, porque unas pocas ventas muy grandes lo jalan hacia arriba.
* **Utilidad.** Va de −6,599.98 a 8,399.98. La mayor perdida es una impresora 3D (Cubify CubeX) en US que se vendio con 70 % de descuento, y la mayor ganancia es una copiadora Canon en US que se vendio sin descuento.
* **Descuento.** Va de 0 a 0.85, asi que ningun descuento pasa del 100 %.
* **¿Son errores?** No los tomamos como errores. Son ventas caras con descuentos altos o sin descuento, que es justo lo que queremos estudiar. Un valor extremo no siempre es un error (Nota 07).

### Tabla de hallazgos

> **Guia:** Para cada señal relevante registren columna o registro, operacion que la detecto, evidencia, posible efecto analitico y aclaracion pendiente. Si no detectan problemas en un aspecto, incluyan la comprobacion que respalda esa observacion.

| Hallazgo | ¿Como lo encontramos? | Evidencia | ¿Afecta la pregunta? | ¿Que hicimos? |
|---|---|---|---|---|
| Descuentos con tres decimales | `value_counts()` de Descuento | 629 ventas, todas Copiers de LATAM | Si, porque cambian el tramo de descuento | Regla 4, se redondean |
| Mismo pedido y producto repetido | `duplicated()` con la clave | 76 filas (38 parejas) | Si, pueden inflar ventas y utilidad | Reglas 5 y 6 |
| Codigos de cliente con minuscula | `str.upper()` | 4 codigos en 139 filas | No, ningun cliente se junta con otro | Regla 2 |
| Nombres de producto con espacios | `str.strip()` | 16 nombres | No | Regla 2 |
| Columnas de codigo guardadas como numero | `dtypes` | 10 columnas `int64` | No, no se usan en calculos | Nada |
| Ganancia repite el signo de Utilidad | Comparar las dos columnas | 0 diferencias | No | Regla 7, se vuelve a calcular |
| Catalogos con claves repetidas | `duplicated()` en Regiones y Productos | 5 paises y 476 codigos repetidos | Si, si se cruzaran se multiplicarian las ventas | Regla 10, no se cruzan |
| Sin datos faltantes | `isna()` y textos vacios | 0 en las 35 columnas | No | Nada |
| Valores extremos | `describe()`, `nsmallest()` y `nlargest()` | La mayor perdida es de −6,599.98 | Si, son parte de lo que estudiamos | Se conservan |

## 3.4 Definicion de reglas de preparacion

> **Guia:** Para cada hallazgo que se tratara, indiquen que se modificara, que regla lo autoriza y como se comprobara. Si no existe respaldo suficiente, conserven el caso como pendiente.



En esta seccion decidimos que hacer con cada hallazgo del perfil (seccion 3.3). Para cada uno anotamos la regla, en que nos apoyamos y como comprobamos que se aplico bien. Las reglas 5 y 7 las revisamos con la profesora el 3 de noviembre de 2026, y ella indico tratarlas asi.
| # | Que encontramos | Regla | Como se comprueba | Fundamento |
|---|---|---|---|---|
| 1 | No hay filas exactamente repetidas | No se retira ninguna fila | `duplicated()` da 0 | Guia av. 2, seccion 3.5, Duplicados |
| 2 | 16 nombres de producto tienen espacios al inicio o al final, y 4 codigos de cliente tienen una letra minuscula (por ejemplo Dp-13240) en 139 filas | Se quitan los espacios de las orillas y los codigos se pasan a mayusculas | Cambian 16 nombres y 139 codigos de cliente. Siguen siendo 1,590 clientes distintos, asi que no hubo cambios en ellos | Guia av. 2, seccion 3.5, Texto; Nota 08, seccion 5 |
| 3 | Las fechas y los numeros tienen el tipo correcto | Se convierten para asegurarlo, sin cambiar valores | Ningun valor queda vacio por la conversion al aplicar errors='coerce' | Guia av. 2, seccion 3.5, Tipos |
| 4 | 629 descuentos con tres decimales (0.002, 0.202, 0.402 y 0.602), todos de Copiers en LATAM | Por indicacion de la profesora se toman como error de captura y se redondean a dos decimales. El valor original se guarda en la columna Descuento original | Cambian exactamente 629 valores. 461 quedan en 0, 41 en 0.2, 104 en 0.4 y 23 en 0.6. Los demas descuentos no cambian | Indicacion de la profesora (3 nov 2026); guia av. 2, seccion 3.5 |
| 5 | 3 parejas con el mismo pedido y producto, pero con cliente, ciudad y fecha distintos | No se pueden tomar como envios del mismo pedido. Se pasan a pendientes con su motivo | Hay 6 filas pendientes | Guia av. 2, seccion 3.5, Duplicados; guia av. 3, secciones 3.3 y 3.4 |
| 6 | 35 parejas con el mismo pedido y producto que solo cambian en cantidad, ventas, utilidad y costo de envio | Por indicacion de la profesora se toman como un pedido enviado en dos partes. Se unen en una fila, se suman Cantidad, Ventas, Utilidad y Costo Envío, y se agrega la columna Número de envíos | Las filas validas no repiten pedido y producto. Los totales de Cantidad, Ventas, Utilidad y Costo Envío son iguales antes y despues | Indicacion de la profesora (3 nov 2026); guia av. 2, seccion 3.5, Duplicados |
| 7 | Ganancia solo dice si la Utilidad es positiva o negativa | como se unificaron algunas filas se vuelve a calcular la ganancia con la nueva utilidad | Coincide el signo de la utilidad con la ganancia | Guia av. 3, seccion 3.3 |
| 8 | La pregunta necesita el mes y el tramo de descuento, que no existen como columnas | Se crean Mes (año y mes de Fecha Salida) y Tramo de descuento (0 %, 1 a 10 %, 11 a 20 %, 21 a 30 %, 31 a 40 %, 41 a 50 % y mas de 50 %) | Todas las filas tienen mes y tramo, y hay 48 meses | Avance 1, seccion 3.7; guia av. 3, seccion 3.2 |
| 9 | El analisis es solo de cuatro mercados | La tabla que hicimos conserva todos los mercados y el analisis centrado en usa US, LATAM, EU y APAC | Nuestra tabla tiene 40,985 filas | Avance 2, seccion 3.1; guia av. 2, seccion 3.7 |
| 10 | La hoja de Ventas ya viene completa y las hojas secundarias (Regiones y Productos) tienen paises y productos repetidos | No se mezclan con Ventas (sin cruces ni merge). Solo se consultan para verificar que ningun pais o producto este mal escrito o inventado | Todos los paises y productos de Ventas si existen en las listas de apoyo (las demas hojas) y la tabla de Ventas conserva sus 51,290 filas sin duplicarse | Guia av. 3, seccion 3.2, "No forzar operaciones"; Nota 09, seccion 5 |
|11| Existen descuentos que son 0% |tomar solo los descuentos mayores a cero |Limpiar los prodectos que tengandescuento cero| vance 2, seccion 3.1; guia av. 2, seccion 3.6  |
|12|Se encontraron 3 ventas raras|eliminar 3 filas errores de captura|cotejando id de ventas ||
|13|Se observa que hay utilidad y costo de venta |no contar el costo de envio en la utilidad | verificando el descuento de las ventas y filtrando datos |
**Tratamientos que no aplican en nuestro caso**

| Aspecto | Se aplica? | Por que |
|---|---|---|
| Faltantes | No | Ninguna de las 35 columnas tiene valores vacios (seccion 3.3, Faltantes) |
| Categorias | No | Mercado, Categoria, Subcategoria y País estan escritos correctamente (seccion 3.3, Diversidad y frecuencias) |
| Fechas | Solo se comprueban | Las fechas tienen un formato correcto. Las ventas van del 1 de enero de 2011 al 31 de diciembre de 2014 y ninguna entrega es anterior a su salida |
| Unidades | No | El archivo no dice en que moneda estan los montos. Por eso se trabaja con el margen, que es una division y no necesita conocer la moneda (avance 1, seccion 3.9) |


### Nota: Fechas lo agregamos porque antes no teniamos claro cuando terminaron las ventas y creiamos que era en 2025. Tras trabajar en el proyecto descubrimos que la fecha 2015 corresponde a una fecha en la que se entregaron algunos productos y que en realidad la fecha en que se dejo de vender es en diciembre de 2014

**Nota:** en la revision del avance 2 la profesora nos pidio dejar solo las reglas de negocio, quitar las 3 parejas con cliente distinto en lugar de dejarlas pendientes y trabajar solo con ventas con descuento. Las reglas nuevas estan en el avance 3 (`notebooks/avance3_etl.ipynb`, seccion 3.3).


## 3.5 Limpieza y normalizacion

> **Guia:** Trabajen sobre una copia y mantengan una referencia a los valores recibidos. Apliquen unicamente los tratamientos pertinentes para su dataset.

Aqui aplicamos las reglas de la seccion 3.4 sobre una copia de la tabla, asi el Excel original no se toca.

In [ ]:

# Revisar los nombres reales de las columnas
print("Columnas del archivo:")
for columna in ventas.columns:
    print(repr(columna))


Columnas del archivo:
'Número Venta'
'ID Venta'
'Mes Salida'
'Dia Salida'
'Año Salida'
'Fecha Salida'
'Dia Semana Fecha Salida'
'Mes Entrega'
'Dia Entrega'
'Año Entrega'
'Fecha Entrega'
'Dia Semana Fecha Entrega'
'Dias Transcurridos'
'Dias laborables transcurridos'
'Método Envio'
'ID Cliente'
'Número Cliente'
'Nombre Cliente'
'Segmento'
'Ciudad'
'Estado'
'País'
'Mercado'
'Region'
'ID Producto'
'Categoria'
'Subcategoria'
'Nombre del producto'
'Ventas'
'Cantidad'
'Descuento'
'Utilidad'
'Ganancia'
'Costo Envío'
'Prioridad Envio'


In [ ]:
import pandas as pd
from pathlib import Path

# Configurar Pandas para que imprima TODAS las columnas sin truncar
pd.set_option('display.max_columns', None)

#Carga reproducible
if 'ventas' not in locals():
    if 'ruta' not in locals():
        archivos = list(Path('.').glob('*.xlsx'))
        if archivos:
            ruta = archivos[0]
        else:
            from google.colab import files
            subido = files.upload()
            ruta = Path(next(iter(subido)))
    ventas = pd.read_excel(ruta, sheet_name="Ventas Supermercado")

#Copia de trabajo (no modificamos el excel original)
ventas_limpias = ventas.copy()

#Normalizar texto
for col in ["ID Venta", "ID Cliente", "ID Producto"]:
    ventas_limpias[col] = ventas_limpias[col].astype("string").str.strip().str.upper()

#Normalizar fechas
for col in ["Fecha Salida", "Fecha Entrega"]:
    ventas_limpias[col] = pd.to_datetime(ventas_limpias[col], errors="coerce")

#Normalizar numeros
num_cols = ["Cantidad", "Ventas", "Descuento", "Utilidad", "Costo Envío"]
for col in num_cols:
    ventas_limpias[col] = pd.to_numeric(ventas_limpias[col], errors="coerce")

#Calcular metricas de envios multiples antes de agrupar
clave = ["ID Venta", "ID Producto"]

#Suma acumulada total del costo de envio por producto en el pedido
ventas_limpias["Costo Envío Total"] = (
    ventas_limpias.groupby(clave, dropna=False)["Costo Envío"].transform("sum")
)

#Conteo de cuantos envios repetidos tuvo ese producto
ventas_limpias["Número de envíos"] = (
    ventas_limpias.groupby(clave, dropna=False)["ID Venta"].transform("size")
)

#Reglas de agregacion completa
reglas = {}
for col in ventas_limpias.columns:
    if col in clave:
        continue
    elif col in ["Cantidad", "Ventas", "Utilidad"]:
        reglas[col] = "sum"            # Se suman cantidades, ventas y utilidades
    elif col == "Costo Envío":
        reglas[col] = "first"          # Costo del primer paquete
    elif col == "Costo Envío Total":
        reglas[col] = "first"          # Suma total de todos los paquetes enviados
    elif col == "Número de envíos":
        reglas[col] = "first"          # Total de paquetes/apariciones
    elif col == "Fecha Salida":
        reglas[col] = "max"
    else:
        reglas[col] = "first"

# Agrupar la base completa (de 51,290 a 51,252 registros)
ventas_agrupadas = ventas_limpias.groupby(clave, dropna=False).agg(reglas).reset_index()

#Recalcular Ganancia
ventas_agrupadas["Ganancia"] = ventas_agrupadas["Utilidad"].apply(
    lambda x: "Ganancia" if x >= 0 else "Perdida"
)


print("=== RESUMEN DE CONTROL ===")
print(f"Filas originales: {len(ventas)}")
print(f"Filas limpias y agrupadas: {len(ventas_agrupadas)}")
print(f"Total registros/paquetes representados: {ventas_agrupadas['Número de envíos'].sum()}\n")

# Seleccionar todas las columnas clave de negocio para imprimir
cols_negocio = [
    "ID Venta", "ID Producto", "Subcategoria",
    "Cantidad", "Ventas", "Descuento", "Utilidad", "Ganancia",
    "Costo Envío", "Costo Envío Total", "Número de envíos"
]

print("MUESTRA DE LOS 38 CASOS CON ENVÍOS MÚLTIPLES ")
display(ventas_agrupadas[ventas_agrupadas["Número de envíos"] > 1][cols_negocio].head(15))

=== RESUMEN DE CONTROL ===
Filas originales: 51290
Filas limpias y agrupadas: 51252
Total registros/paquetes representados: 51290

MUESTRA DE LOS 38 CASOS CON ENVÍOS MÚLTIPLES 


,ID Venta,ID Producto,Subcategoria,Cantidad,Ventas,Descuento,Utilidad,Ganancia,Costo Envío,Costo Envío Total,Número de envíos
214,AL-2014-4840,TEC-MEM-10002524,Accessories,4,113.520,0.0,21.480,Ganancia,5.43,8.20,2
1052,AU-2011-54176,OFF-BI-10000168,Binders,6,30.780,0.1,5.040,Ganancia,0.70,1.34,2
2426,AU-2013-52804,OFF-AP-10000729,Appliances,6,2840.832,0.1,725.832,Ganancia,196.63,262.20,2
2577,AU-2013-66265,OFF-AR-10003461,Art,2,46.872,0.1,19.752,Ganancia,6.68,7.53,2
3164,AU-2014-34317,OFF-BI-10003309,Binders,6,57.510,0.1,9.450,Ganancia,2.12,2.98,2
4006,BA-2011-71739,OFF-LA-10002806,Labels,7,79.590,0.0,38.850,Ganancia,5.29,5.78,2
4024,BA-2011-77892,TEC-PH-10001751,Phones,7,4473.000,0.0,1610.280,Ganancia,34.00,57.95,2
4202,BE-2012-2740636,OFF-AR-10004884,Art,6,74.160,0.0,20.700,Ganancia,2.98,4.82,2
7128,CH-2012-137561,FUR-CH-10000892,Chairs,4,1230.480,0.0,24.560,Ganancia,68.73,123.64,2
8450,CH-2014-45174,FUR-BO-10004529,Bookcases,6,1008.180,0.0,120.960,Ganancia,62.77,74.12,2


### reglas 1 y 2

Comprobamos que el Excel sigue igual y que no hay filas exactamente repetidas.

In [ ]:
# Regla 1. El Excel original sigue igual, su huella debe empezar con a599dc2cf5d00385
huella_actual = hashlib.sha256(Path(ruta).read_bytes()).hexdigest()
print("El Excel original sigue igual?", huella_actual[:16] == "a599dc2cf5d00385")
# Regla 2. No hay filas exactamente repetidas asi que no se quita ninguna
print("Filas exactamente repetidas:", ventas.duplicated().sum())

El Excel original sigue igual? True
Filas exactamente repetidas: 0


### Texto y tipos (reglas 3 y 4)

Quitamos los espacios de las orillas en los codigos y en los nombres de producto, y pasamos los codigos a mayusculas. Revisamos que al hacerlo ningun cliente se junte con otro. Despues convertimos fechas y numeros a su tipo y revisamos que ningun valor se haya vuelto vacio por la conversion.

In [ ]:
# Regla 3. quitar espacios al inicio y al final y pasar los codigos a mayusculas
codigos = ["ID Venta", "ID Cliente", "ID Producto"]
for col in codigos:
    ventas_limpias[col] = ventas_limpias[col].astype("string").str.strip().str.upper()
    print(f"{col}: valores que cambiaron:", (ventas_limpias[col] != ventas[col]).sum())

# 4 codigos de cliente tenian una minuscula, por ejemplo Dp-13240, que ahora queda como DP-13240.
# Revisamos que con ese cambio no se junten dos clientes distintos en uno solo.
# Si el numero de clientes es el mismo antes y despues, no se junto ninguno.
print("Clientes distintos antes:", ventas["ID Cliente"].nunique(), "y después:", ventas_limpias["ID Cliente"].nunique())

# En los nombres de producto solo quitamos los espacios de las orillas sin cambiar a mayusculas,
# y contamos cuantos nombres cambiaron.
ventas_limpias["Nombre del producto"] = ventas_limpias["Nombre del producto"].str.strip()
print("Nombre del producto: valores que cambiaron:", (ventas_limpias["Nombre del producto"] != ventas["Nombre del producto"]).sum())

# Regla 4. Tipos: Las fechas y numeros ya venian con el tipo correcto. Se convierten para asegurarlo.
# errors="coerce" deja vacio como null lo que no se pudo convertir, si al contarlo saliera un null quiere decir que hay algun
#valor que no se pudo convertir asi que tiene un error de formato.
fechas = ["Fecha Salida", "Fecha Entrega"]
numeros = ["Cantidad", "Ventas", "Descuento", "Utilidad", "Costo Envío"]
for col in fechas:
    ventas_limpias[col] = pd.to_datetime(ventas_limpias[col], errors="coerce")
for col in numeros:
    ventas_limpias[col] = pd.to_numeric(ventas_limpias[col], errors="coerce")
print("\nValores que quedaron vacíos al convertir:", ventas_limpias[fechas + numeros].isna().sum().sum())
print("Filas:", len(ventas_limpias))

ID Venta: valores que cambiaron: 0
ID Cliente: valores que cambiaron: 139
ID Producto: valores que cambiaron: 0
Clientes distintos antes: 1590 y después: 1590
Nombre del producto: valores que cambiaron: 16

Valores que quedaron vacíos al convertir: 0
Filas: 51290


### Convertir los descuentos a dos decimales (regla 5)
Guardamos el descuento original en otra columna y redondeamos a dos decimales. Asi se puede ver que valor tenia cada venta antes del cambio.



In [ ]:
# Regla 5. Se guarda el original y se redondea a dos decimales
ventas_limpias["Descuento original"] = ventas_limpias["Descuento"]
ventas_limpias["Descuento"] = ventas_limpias["Descuento"].round(2)

# ¿Cuantos cambiaron y a que valor quedaron?
redondeados = ventas_limpias["Descuento"] != ventas_limpias["Descuento original"]
print("Descuentos redondeados:", redondeados.sum())
print(ventas_limpias.loc[redondeados, ["Descuento original", "Descuento"]].round(3).value_counts())

# ¿Todos los redondeados son Copiers de LATAM?
print("\nMercado y subcategoria de los redondeados:")
print(ventas_limpias.loc[redondeados, ["Mercado", "Subcategoria"]].value_counts())

Descuentos redondeados: 629
Descuento original  Descuento
0.002               0.0          461
0.402               0.4          104
0.202               0.2           41
0.602               0.6           23
Name: count, dtype: int64

Mercado y subcategoria de los redondeados:
Mercado  Subcategoria
LATAM    Copiers         629
Name: count, dtype: int64


In [ ]:
# ---------------------------------------------------------
# REGLA 5. SEPARACION DE PAREJAS INCONSISTENTES (6 FILAS -> PENDIENTES)
# ---------------------------------------------------------
clave = ["ID Venta", "ID Producto"]

# 1. Identificar las filas donde coinciden pedido y producto
repetidas = ventas_limpias[ventas_limpias.duplicated(subset=clave, keep=False)]

# 2. Revisar cuales cambian en cliente, ciudad o fecha de salida
columnas_revision = ["ID Cliente", "Ciudad", "Fecha Salida"]
parejas_inconsistentes = repetidas.groupby(clave)[columnas_revision].nunique() > 1

# 3. Obtener la lista de parejas inconsistentes en formato limpio
parejas_malas = parejas_inconsistentes[parejas_inconsistentes.any(axis=1)].reset_index()[clave]

# 4. Cruzar para obtener las 6 filas con inconsistencias
pendientes_duplicados = ventas_limpias.merge(parejas_malas, on=clave, how="inner").copy()
pendientes_duplicados["Motivo_Pendiente"] = "Inconsistencia en Cliente, Ciudad o Fecha en pareja con mismo ID Venta e ID Producto"

# 5. Filtrar las filas validas (excluyendo las 6 inconsistentes)
idx_pendientes = ventas_limpias.set_index(clave).index.isin(parejas_malas.set_index(clave).index)
ventas_validas = ventas_limpias[~idx_pendientes].copy()

# ---------------------------------------------------------
# REGLA 6. CONSOLIDACION DE LAS 35 PAREJAS VALIDAS (70 FILAS)
# ---------------------------------------------------------
ventas_validas["Costo Envío Total"] = ventas_validas.groupby(clave, dropna=False)["Costo Envío"].transform("sum")
ventas_validas["Número de envíos"] = ventas_validas.groupby(clave, dropna=False)["ID Venta"].transform("size")

reglas_agg = {}
for col in ventas_validas.columns:
    if col in clave:
        continue
    elif col in ["Cantidad", "Ventas", "Utilidad"]:
        reglas_agg[col] = "sum"
    elif col in ["Costo Envío", "Costo Envío Total", "Número de envíos"]:
        reglas_agg[col] = "first"
    elif col == "Fecha Salida":
        reglas_agg[col] = "max"
    else:
        reglas_agg[col] = "first"

ventas_agrupadas = ventas_validas.groupby(clave, dropna=False).agg(reglas_agg).reset_index()

ventas_agrupadas["Ganancia"] = ventas_agrupadas["Utilidad"].apply(
    lambda x: "Ganancia" if x >= 0 else "Perdida"
)

print("=== VERIFICACIÓN DE CONTROL ===")
print(f"Filas originales recibidas: {len(ventas)}")
print(f"Regla 5 - Filas enviadas a pendientes: {len(pendientes_duplicados)} (Esperado: 6)")
print(f"Regla 6 - Filas finales consolidadas: {len(ventas_agrupadas)} (Esperado: 51,249)")
print(f"Duplicados en clave candidata tras consolidar: {ventas_agrupadas.duplicated(subset=clave).sum()} (Esperado: 0)")


=== VERIFICACIÓN DE CONTROL ===
Filas originales recibidas: 51290
Regla 5 - Filas enviadas a pendientes: 6 (Esperado: 6)
Regla 6 - Filas finales consolidadas: 51249 (Esperado: 51,249)
Duplicados en clave candidata tras consolidar: 0 (Esperado: 0)


## 3.6 Validacion de los datos preparados

> **Guia:** Conviertan las reglas pertinentes en comprobaciones ejecutables. Cada validacion debe expresar el uso al que se aplica, la condicion esperada y la accion prevista si falla.

Esta parte la hicimos en el avance 3, donde las reglas se volvieron preguntas de si o no para cada fila (`notebooks/avance3_etl.ipynb`, paso 5 de la seccion 3.3 y seccion 3.4).

## 3.7 Comparacion e interpretacion final

> **Guia:** Comparen el perfil inicial con el resultado preparado. Concluyan que datos pueden utilizarse para el analisis previsto, bajo que condiciones y que problemas requieren consulta o revision.

Esta parte la hicimos en el avance 3, donde comparamos el Excel con los datos limpios y contamos a donde fue cada fila (`notebooks/avance3_etl.ipynb`, secciones 3.4 y 3.5).

## 3.8 Reproducibilidad y trabajo en equipo

> **Guia:** Comprueben que el notebook puede ejecutarse desde el inicio. Indiquen dependencias, rutas y como obtener los datos. Incluyan la bitacora breve del equipo.

Esta parte la hicimos en el avance 3. Como correr todo y que se necesita esta en el `README.md`, la prueba de que siempre sale lo mismo esta en `notebooks/avance3_etl.ipynb` (seccion 3.7) y la bitacora del equipo esta en `bitacora/bitacora_equipo.md`.

### Bitacora de transformaciones

| Paso | ¿Que hicimos? | Filas antes → despues | ¿Como lo comprobamos? |
|---|---|---|---|
| Copia de trabajo | Copiamos `ventas` en `ventas_limpias` para no tocar el original | 51,290 → 51,290 | La huella del Excel sigue igual |
| Texto | Quitamos espacios y pasamos a mayusculas los codigos, y quitamos espacios en los nombres de producto | 51,290 → 51,290 | Cambiaron 139 codigos de cliente y 16 nombres, y siguen siendo 1,590 clientes |
| Tipos | Convertimos fechas y montos para asegurar su tipo | 51,290 → 51,290 | Ningun valor quedo vacio |
| Descuentos | Redondeamos a dos decimales los 629 descuentos raros y guardamos el original | 51,290 → 51,290 | 461 quedaron en 0, 41 en 0.2, 104 en 0.4 y 23 en 0.6 |
| Parejas con cliente distinto | Mandamos las 3 parejas (6 filas) a pendientes | 51,290 → 51,284 | Hay 6 filas en pendientes |
| Juntar envios | Juntamos las 35 parejas sumando cantidad, ventas y utilidad. En `Costo Envío` queda el del primer envio y en `Costo Envío Total` la suma, y agregamos `Número de envíos` | 51,284 → 51,249 | Ya no se repite ningun pedido + producto |
| Ganancia | La volvimos a calcular con la utilidad ya sumada | 51,249 → 51,249 | Coincide con el signo de la utilidad |

En el avance 3 este proceso lo pasamos a funciones y la bitacora ya se genera sola (`bitacora/bitacora.csv`).